# Qwen3-VL-4B のベース記録を Colab で取る

手順:
1. ランタイム → ランタイムのタイプを変更 → GPU（A100 が選べればそれ、無ければ L4）
2. 事前に Google Drive の `MyDrive/movie_llm/` に `frames_f32_s640.zip` と `frames_f16_s640.zip` を置く（手元で `scripts/jvqa_cache_frames.py --zip` で作ったもの）
3. 上から順にセルを実行する。予測は `MyDrive/movie_llm/results/<name>/predictions.jsonl` に書かれ、途中で切れても再実行すれば続きから進む
4. 終わったら `predictions.jsonl` を手元に持ち帰り、`scripts/jvqa_eval.py` で採点する（GPT-4o 採点は手元の .env を使う）


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
# Colab には torch が入っているので、それ以外を入れる（transformers 5.x が必要）
!pip install -q "transformers>=5.0" "accelerate>=1.0" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip install -q -e . --no-deps
import transformers, av; print("transformers", transformers.__version__, "av", av.__version__)

## 2. Drive をマウントしてフレームキャッシュを展開

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
import os; os.makedirs(f'{DRIVE}/results', exist_ok=True)
!ls -la {DRIVE}

In [ ]:
# zip をローカルディスクに展開する（Drive 上の小さなファイルを直接読むと遅い）
!mkdir -p /content/frames
!unzip -q -o {DRIVE}/frames_f32_s640.zip -d /content/frames
!unzip -q -o {DRIVE}/frames_f16_s640.zip -d /content/frames
!ls /content/frames | wc -l
!du -sh /content/frames

## 3. 動画ファイルのダミーを作る

予測スクリプトは「動画ファイルがあるか」で対象を決める。Colab には動画が無いので、フレームキャッシュがある動画と同じ名前の空ファイルを置く。フレームはキャッシュから読まれるので、動画の中身は使われない。

In [ ]:
import os
os.makedirs('/content/videos', exist_ok=True)
n = 0
for vid in os.listdir('/content/frames'):
    open(f'/content/videos/{vid}.mp4', 'a').close(); n += 1
print('動画（ダミー）', n)

## 4. 4B で予測（32フレーム → 16フレーム）

In [ ]:
%cd /content/movie_llm
# 32フレーム。A100 で 30〜60 分の見込み。途中で切れたら同じセルを再実行する
!python /content/movie_llm/scripts/jvqa_predict.py --name qwen3vl-4b_f32 --model Qwen/Qwen3-VL-4B-Instruct --frames 32 \
    --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results --type multi_choice
!python /content/movie_llm/scripts/jvqa_predict.py --name qwen3vl-4b_f32 --model Qwen/Qwen3-VL-4B-Instruct --frames 32 \
    --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results --type yes_no
!python /content/movie_llm/scripts/jvqa_predict.py --name qwen3vl-4b_f32 --model Qwen/Qwen3-VL-4B-Instruct --frames 32 \
    --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results --type open

In [ ]:
%cd /content/movie_llm
# 16フレーム（手元の 2B と同条件）
for t in ['multi_choice', 'yes_no', 'open']:
    !python /content/movie_llm/scripts/jvqa_predict.py --name qwen3vl-4b_f16 --model Qwen/Qwen3-VL-4B-Instruct --frames 16 \
        --no-fetch --video-dir /content/videos --frames-dir /content/frames --out-dir {DRIVE}/results --type {t}

## 5. 規則採点だけここで確認（4択・yes_no）。open の GPT-4o 採点は手元で行う

In [ ]:
for name in ['qwen3vl-4b_f32', 'qwen3vl-4b_f16']:
    !wc -l {DRIVE}/results/{name}/predictions.jsonl
    !python /content/movie_llm/scripts/jvqa_eval.py --predictions {DRIVE}/results/{name}/predictions.jsonl --name {name} 2>/dev/null | grep -E "^\| (multi_choice|yes_no|count|temporal|spatial|action|causal|short|medium) "
print('手元での採点:')
print('  scp などで results/<name>/predictions.jsonl を持ち帰り、')
print('  uv run python scripts/jvqa_eval.py --predictions results/<name>/predictions.jsonl --name <name> --judge --judge-only-open \\')
print('      --judge-base-url https://ai-gateway.vercel.sh/v1 --judge-api-key-env AI_GATEWAY_API_KEY --judge-model openai/gpt-4o')